In [ ]:
# install latest version from local files (only necessary after pulling updated version)
%pip install -e ..

In [ ]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
# from biopap.modules.segmentation import Segmentation

In [ ]:
import matplotlib.pyplot as plt

### Loading data

In [ ]:
bdf_file = '/data/BioPAP/data/CBN1BD071212-Cropped.bdf'

In [ ]:
data = LoadData(bdf_file).raw
print(data.info)

In [ ]:
data.plot()

### Anonymization

In [ ]:
# defaults: Participant='0'; Experimenter='mne_anonymize'
ano = Anonymize(raw=data, daysback=100, experimenter='J. Jansen', participant_id='AB00123')
ano.run()
ano.processed.info

### Rereferencing

In [ ]:
ref = Rereference(raw=ano.processed, ref_channels=['EXG1', 'EXG2'])
ref.run()
ref.processed.info

### hEOG & vEOG

In [ ]:
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])
eog.run()

In [ ]:
print(eog.processed.info)

### Filtering

In [ ]:
flt = IIRFilter(
    raw=eog.processed,
    l_freq=0.1,
    h_freq=35)
flt.run()

In [ ]:
flt.processed.plot()

### Downsampling

In [ ]:
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)
dwn.run()

In [ ]:
print(dwn.processed)
print(dwn.events)
dwn.processed.plot()